# Create African Academy of Sciences ARISE Awards

Creates OpenAlex award rows from the African Academy of Sciences (AAS) African Research Initiative for Scientific Excellence (ARISE) official grantee portal.

**Prerequisites:**
- Admin/human with AWS credentials runs `scripts/local/aas_arise_to_s3.py` to download the official ARISE JSON API response, write parquet, and upload it.
- Contractor has prepared the script, notebook, registry entry, tracker row, and local QA, but does not have S3/Databricks access.
- The downloader writes all parquet columns as strings per `plans/awards/how-to-add-a-funder.md`; this notebook does type conversion with `TRY_CAST` / `TRY_TO_DATE`.

**Data source:** `https://arise-api.aasciences.app/api/grantees`  
**Public grantee pages:** `https://arise.aasciences.app/grantees/{slug}`  
**Programme page:** `https://arise.aasciences.app/`  
**Programme amount note:** `https://arise.aasciences.app/news/major-contributions-announced-for-arise-programme-to-advance-scientific-excellence-in-africa`  
**S3 location:** `s3a://openalex-ingest/awards/aas_arise/aas_arise_grantees.parquet`  
**Local full-source validation on 2026-05-27:** 47 grantee/project rows, 47 unique native IDs, 100% coverage for title, description, grantee name, institution, country, thematic area, registered date, and landing page; one 2024 cohort across 38 countries.

**AAS funder:**
- funder_id: 4320327323
- display_name: "African Academy of Sciences"
- country: KE

**Source-shape note:** ARISE is implemented by AAS with AU/EU partners, and the official ARISE site lists grantee research projects. The programme amount page says ARISE supports 47 research teams with grants of up to EUR 500,000 per team, but the grantee API does not publish exact per-team awarded budgets. This notebook preserves the source ceiling columns in the raw table and intentionally leaves OpenAlex `amount`/`currency` NULL with a section 6.7 waiver, following fellowship/grantee-directory precedents where per-recipient budgets are not public.

**Mapping summary:** One OpenAlex award row per ARISE grantee project. Native key is `aas-arise-{slug}`. `display_name` is the API `research_title`; `description` combines problem statement, progress highlights, key findings, and potential impact. `lead_investigator` is the named grantee, with affiliation from the API institution and country. `funding_type='grant'`; `funder_scheme` combines ARISE, cohort, and thematic area.

## Step 1: Create Raw Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.aas_arise_raw
USING delta
AS
SELECT
    *,
    current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/aas_arise/aas_arise_grantees.parquet`;

In [ ]:
%sql
-- Check raw row count. Local validation on 2026-05-27 found 47 rows.
SELECT COUNT(*) as total_aas_arise_grantees
FROM openalex.awards.aas_arise_raw;

In [ ]:
%sql
-- Verify actual column names before transform logic references them.
DESCRIBE openalex.awards.aas_arise_raw;

In [ ]:
%sql
-- Sample raw ARISE data.
SELECT
    funder_award_id,
    source_slug,
    display_name,
    grantee_name,
    institution,
    country,
    cohort,
    thematic_area,
    registered_year,
    amount,
    currency,
    program_ceiling_amount_eur,
    landing_page_url
FROM openalex.awards.aas_arise_raw
LIMIT 10;

## Step 1.5: Inspect Raw Data, Money, Dates, and Native Keys

In [ ]:
%sql
-- Money-field scan per runbook Step 1.5. The programme ceiling columns are not per-award budgets.
SELECT column_name
FROM openalex.information_schema.columns
WHERE table_schema = 'awards'
  AND table_name = 'aas_arise_raw'
  AND LOWER(column_name) RLIKE
    'amount|amt|total|value|sum|funded|funding|cost|budget|grant|award|eur|currency';

In [ ]:
%sql
-- Confirm amount/date coverage before mapping.
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS rows_with_per_award_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_with_per_award_amount,
    COUNT(program_ceiling_amount_eur) AS rows_with_program_ceiling,
    ROUND(try_divide(COUNT(program_ceiling_amount_eur), COUNT(*)) * 100.0, 1) AS pct_with_program_ceiling,
    MIN(TRY_CAST(program_ceiling_amount_eur AS DOUBLE)) AS min_program_ceiling_eur,
    MAX(TRY_CAST(program_ceiling_amount_eur AS DOUBLE)) AS max_program_ceiling_eur,
    COUNT(start_date) AS rows_with_start_date,
    MIN(TRY_CAST(start_year AS INT)) AS min_year,
    MAX(TRY_CAST(start_year AS INT)) AS max_year
FROM openalex.awards.aas_arise_raw;

In [ ]:
%sql
-- Native-key inspection. Source slug is unique and is used in funder_award_id.
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT source_slug) AS distinct_source_slugs,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    COUNT(DISTINCT grantee_name) AS distinct_grantee_names,
    COUNT(DISTINCT display_name) AS distinct_research_titles
FROM openalex.awards.aas_arise_raw;

In [ ]:
%sql
-- Source cohort and thematic-area distributions.
SELECT
    cohort,
    thematic_area,
    COUNT(*) AS cnt,
    COUNT(DISTINCT country) AS distinct_countries
FROM openalex.awards.aas_arise_raw
GROUP BY cohort, thematic_area
ORDER BY cnt DESC;

In [ ]:
%sql
-- Country distribution from the official grantee API.
SELECT country, COUNT(*) AS cnt
FROM openalex.awards.aas_arise_raw
GROUP BY country
ORDER BY cnt DESC, country
LIMIT 50;

In [ ]:
%sql
-- Document the amount decision that will be applied in Step 2.
SELECT
    program_amount_raw,
    program_ceiling_amount_eur,
    program_ceiling_currency,
    amount_decision,
    amount_source_url,
    COUNT(*) AS rows
FROM openalex.awards.aas_arise_raw
GROUP BY program_amount_raw, program_ceiling_amount_eur, program_ceiling_currency, amount_decision, amount_source_url;

## Step 1.6: Funder Existence Fail-Fast

In [ ]:
%sql
-- Must return exactly 1 row. If this is empty, STOP: the funder is missing from openalex.common.funder.
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320327323;

## Step 2: Transform to OpenAlex Award Schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.aas_arise_awards
USING delta
AS
WITH
aas_funder AS (
    SELECT
        funder_id,
        display_name,
        ror_id,
        doi
    FROM openalex.common.funder
    WHERE funder_id = 4320327323
),

raw_prepared AS (
    SELECT
        *,
        LOWER(TRIM(funder_award_id)) AS native_award_id,
        TRY_TO_DATE(start_date, 'yyyy-MM-dd') AS parsed_start_date,
        TRY_CAST(start_year AS INT) AS parsed_start_year
    FROM openalex.awards.aas_arise_raw
    WHERE funder_award_id IS NOT NULL
      AND TRIM(funder_award_id) != ''
      AND display_name IS NOT NULL
      AND TRIM(display_name) != ''
),

awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', g.native_award_id))) % 9000000000 as id,

        TRIM(g.display_name) as display_name,

        CASE
            WHEN g.description IS NULL OR TRIM(g.description) = '' THEN NULL
            ELSE TRIM(g.description)
        END as description,

        f.funder_id,
        g.native_award_id as funder_award_id,

        CAST(NULL AS DOUBLE) as amount,
        CAST(NULL AS STRING) as currency,

        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name,
            f.ror_id,
            f.doi
        ) as funder,

        'grant' as funding_type,

        NULLIF(TRIM(g.funder_scheme), '') as funder_scheme,

        'aas_arise_grantees' as provenance,

        g.parsed_start_date as start_date,
        CAST(NULL AS DATE) as end_date,
        COALESCE(YEAR(g.parsed_start_date), g.parsed_start_year) as start_year,
        CAST(NULL AS INT) as end_year,

        struct(
            NULLIF(TRIM(g.lead_investigator_given_name), '') as given_name,
            NULLIF(TRIM(g.lead_investigator_family_name), '') as family_name,
            CAST(NULL AS STRING) as orcid,
            g.parsed_start_date as role_start,
            struct(
                NULLIF(TRIM(g.institution), '') as name,
                NULLIF(TRIM(g.country), '') as country,
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
            ) as affiliation
        ) as lead_investigator,

        CAST(NULL AS STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,

        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,

        NULLIF(TRIM(g.landing_page_url), '') as landing_page_url,
        CAST(NULL AS STRING) as doi,

        concat('https://api.openalex.org/works?filter=awards.id:G', abs(xxhash64(CONCAT(f.funder_id, ':', g.native_award_id))) % 9000000000) as works_api_url,

        current_timestamp() as created_date,
        current_timestamp() as updated_date

    FROM raw_prepared g
    CROSS JOIN aas_funder f
)

SELECT * FROM awards_transformed;

## Step 3: Delete Previous Source Rows and Insert Priority 134

In [ ]:
%sql
-- Remove previous AAS ARISE data before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'aas_arise_grantees' AND priority = 134;

-- Insert into openalex_awards_raw with exact OpenAlex awards schema plus priority.
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id,
    display_name,
    description,
    funder_id,
    funder_award_id,
    amount,
    currency,
    funder,
    funding_type,
    funder_scheme,
    provenance,
    start_date,
    end_date,
    start_year,
    end_year,
    lead_investigator,
    co_lead_investigator,
    investigators,
    landing_page_url,
    doi,
    works_api_url,
    created_date,
    updated_date,
    134 as priority
FROM openalex.awards.aas_arise_awards;

## Handoff/Admin Notes

Contractor-complete handoff stops here. Admin must upload the parquet to S3, run this notebook in Databricks, inspect the Step 6 verification cells, and only then decide whether to add scheduled job YAML. No job YAML is included in this PR.

## Step 6: Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_aas_arise_awards
FROM openalex.awards.aas_arise_awards;

In [ ]:
%sql
-- Confirm the transformed table has the OpenAlex awards columns only.
DESCRIBE openalex.awards.aas_arise_awards;

In [ ]:
%sql
-- Sample transformed awards.
SELECT
    id,
    display_name,
    funder_award_id,
    funder_scheme,
    funding_type,
    amount,
    currency,
    start_date,
    end_date,
    lead_investigator.given_name AS pi_given_name,
    lead_investigator.family_name AS pi_family_name,
    lead_investigator.affiliation.name AS institution,
    lead_investigator.affiliation.country AS country,
    landing_page_url
FROM openalex.awards.aas_arise_awards
LIMIT 10;

In [ ]:
%sql
-- Check ID and native-key uniqueness.
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT id) AS distinct_openalex_ids,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids
FROM openalex.awards.aas_arise_awards;

In [ ]:
%sql
-- Check funder distribution. Should be only African Academy of Sciences F4320327323.
SELECT funder.display_name, funder_id, COUNT(*) as cnt
FROM openalex.awards.aas_arise_awards
GROUP BY funder.display_name, funder_id
ORDER BY cnt DESC;

In [ ]:
%sql
-- Data completeness.
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(start_date) as has_start_date,
    COUNT(end_date) as has_end_date,
    COUNT(landing_page_url) as has_url,
    COUNT(lead_investigator.family_name) as has_pi_family,
    COUNT(lead_investigator.affiliation.name) as has_institution,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) as pct_with_amount,
    ROUND(try_divide(COUNT(description), COUNT(*)) * 100.0, 1) as pct_description,
    ROUND(try_divide(COUNT(lead_investigator.family_name), COUNT(*)) * 100.0, 1) as pct_pi_family,
    ROUND(try_divide(COUNT(lead_investigator.affiliation.name), COUNT(*)) * 100.0, 1) as pct_institution
FROM openalex.awards.aas_arise_awards;

In [ ]:
%sql
-- Amount/currency waiver check. ARISE publishes programme ceiling only, not exact per-team budgets.
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS rows_with_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies,
    MIN(amount) AS min_amount,
    MAX(amount) AS max_amount,
    AVG(amount) AS avg_amount
FROM openalex.awards.aas_arise_awards;

In [ ]:
%sql
-- Year and scheme distribution.
SELECT start_year, funder_scheme, COUNT(*) as cnt
FROM openalex.awards.aas_arise_awards
WHERE start_year IS NOT NULL
GROUP BY start_year, funder_scheme
ORDER BY start_year DESC, cnt DESC;

In [ ]:
%sql
-- Country distribution carried through the lead investigator affiliation.
SELECT lead_investigator.affiliation.country AS country, COUNT(*) AS cnt
FROM openalex.awards.aas_arise_awards
GROUP BY lead_investigator.affiliation.country
ORDER BY cnt DESC, country
LIMIT 50;

In [ ]:
%sql
-- Verify rows inserted into the raw awards table at priority 134.
SELECT
    priority,
    provenance,
    COUNT(*) as cnt,
    COUNT(DISTINCT funder_award_id) as distinct_funder_award_ids,
    COUNT(amount) as rows_with_amount
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'aas_arise_grantees' AND priority = 134
GROUP BY priority, provenance;